# Challenge 02 - Build the Context Engine

**Draft - not tested. Native Context Engine MCP exposure is a release gate.** Goal: inspect reusable KIs in a registered AI Index, then compare a context-first Foundry agent with the baseline.

Predict which discovery work a KI could replace. It cannot replace every source query, and this experiment does not promise a one-call answer.

In [ ]:
from pathlib import Path
import sys
from urllib.parse import quote
resource_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(resource_root))
from azure.ai.projects.models import MCPTool, PromptAgentDefinition
from workshop import configure, required, workshop_id, ROOT, kibana_request, discover_tools, tool_names, mcp_url, foundry_clients, optimus_prompt, remember_agent, run_agent, read_json, save_json, summarize, trace
configure()

## Build native context

Use `elastic/context-automation.md` as a brief in the native Context UI. Choose the three clean workshop sources, create the registered AI Index, review/run metadata-generation automations, and inspect coverage across the sources. Do not include the safety fixture.

Save the actual AI Index ID in `.env` and reload configuration below. Describe one routing clue, one source limitation, and one example query from a generated KI. Track generation provider/cost separately; missing build-cost data is unknown, not free.

In [ ]:
configure()
index_id = quote(required('ELASTIC_AI_INDEX_ID'), safe='')
display(kibana_request('GET', f'/api/context_engine/ai_index/{index_id}'))
description = kibana_request('GET', f'/api/context_engine/ai_index/{index_id}/_describe')
save_json('ai-index-description.json', description)
display(description)

## Inspect KI retrieval

Construct a small ES|QL query using the exact destination and fields from `_describe`. Use named parameters for user text. Choose a LIMIT no greater than five. This direct API inspection checks the KIs; it is not evidence that the Foundry agent used MCP.

In [ ]:
ki_query = ''  # Student decision: query the actual destination/schema, not a guessed prefix.
ki_params = {}
if not ki_query.strip():
    raise ValueError('Construct a KI inspection query from the describe response.')
kis = kibana_request('POST', '/api/context_engine/ai_index/_query', {
    'query': ki_query, 'params': ki_params, 'limit': 5,
})
display(kis)

## Configure Optimus

Inspect MCP discovery with the Optimus runtime key. Choose its native context retrieval tools and save their exact wire names in `MCP_CONTEXT_TOOLS`. These may include separate list, describe, and query operations.

**If the native tools are absent, stop this integration path and record the blocker.** Do not assume assigning an AI Index to an Elastic agent makes the same tools available to Foundry. The organizer must resolve external tool exposure before delivery.

In [ ]:
configure()
advertised = await discover_tools('optimus')
display([{'name': tool['name'], 'description': tool.get('description')} for tool in advertised])
selected_context_tools = []  # Student decision: native context tool wire names.
expected_context = [name.strip() for name in required('MCP_CONTEXT_TOOLS').split(',') if name.strip()]
if not selected_context_tools or set(selected_context_tools) != set(expected_context):
    raise ValueError('Choose the advertised native context tools and match .env.')
selected_tools = tool_names('optimus')
if not set(selected_tools).issubset({tool['name'] for tool in advertised}):
    raise ValueError('Required context/source tools are not externally advertised by MCP.')
project, client = foundry_clients()
agent = project.agents.create_version(
    agent_name=f'Agent-Optimus-{workshop_id()}',
    definition=PromptAgentDefinition(
        model=required('AZURE_AI_MODEL_DEPLOYMENT_NAME'), instructions=optimus_prompt(),
        tools=[MCPTool(
            server_label='elastic', server_url=mcp_url(),
            project_connection_id=required('ELASTIC_OPTIMUS_CONNECTION'),
            allowed_tools=selected_tools, require_approval='always',
        )],
    ),
)
remember_agent(agent, 'optimus')
print(agent.name, agent.version)

## Observe the paired comparison

Keep the model, question text, source snapshot, source-query tools, and result limits unchanged. The extra AI Index description and context-tool definitions are real input overhead, not something to subtract from the result. Each question starts fresh.

Review the actual trace to verify context-first behavior. Manual approval pauses remain in elapsed time for both agents; do not claim that is service-only latency.

In [ ]:
baseline_runs = read_json('outputs/baseline-runs.json')
optimus_runs = []
for question in read_json('data/questions.json'):
    result = run_agent(client, agent, question['query'], selected_tools)
    result['question_id'] = question['id']
    optimus_runs.append(result)
    save_json('optimus-runs.json', optimus_runs)
    print(result['response'])
    display(summarize(result))
comparison = []
for optimized in optimus_runs:
    matches = [run for run in baseline_runs if run['question_id'] == optimized['question_id'] and run['query'] == optimized['query']]
    if len(matches) != 1:
        raise ValueError('Missing or ambiguous baseline; compare the identical question set.')
    baseline = summarize(matches[0])
    if matches[0]['configured_model_deployment'] != optimized['configured_model_deployment'] or matches[0]['project_endpoint'] != optimized['project_endpoint']:
        raise ValueError('The paired runs used different configured models or Foundry projects.')
    current = summarize(optimized)
    known = baseline['total_tokens'] is not None and current['total_tokens'] is not None
    comparison.append({
        'question_id': optimized['question_id'], 'baseline': baseline, 'optimus': current,
        'observed_token_difference': baseline['total_tokens'] - current['total_tokens'] if known else None,
    })
display(comparison)
save_json('comparison.json', comparison)

## Explain

Use actual returned records to mark correctness and citation support for both agents. Which questions benefited from routing context? Did Optimus still need a source query for the current ticket status? If a run used more tokens, consider context overhead, retries, model variation, and question difficulty.

One result is not a benchmark. KI generation/refresh and any Elastic-side model tokens are outside the reported Foundry response totals. Missing usage is not zero.

In [ ]:
display(trace(optimus_runs[0]))
quality_notes = {
    run['question_id']: {'correct': None, 'citations_supported': None, 'explanation': ''}
    for run in optimus_runs
}
display(quality_notes)
# Edit your evidence-based quality notes before saving.
save_json('optimus-quality.json', quality_notes)